# Frozen candidate validation-only CUDA retraining

This notebook validates 30 reusable anchor runs and trains only 20 new strict/compact consensus runs on Colab CUDA. It never evaluates held-out test data and never copies prior checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL='https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPO_DIR=Path('/content/VitalDB-Feature-Selection')
ROOT=Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
DATASET_DIR=ROOT/'data/modeling/full'
GROUP_ROOT=ROOT/'outputs/group_retraining_validation_only'
GROUP_ANALYSIS_DIR=GROUP_ROOT/'analysis'
SELECTION_DIR=ROOT/'outputs/predictive_feature_selection_30s'
CANDIDATE_PATH=SELECTION_DIR/'candidate_subsets.json'
OUTPUT_ROOT=ROOT/'outputs/frozen_candidate_retraining_validation_only'
RUN_FULL_TRAINING=False
CONFIRMATION_TEXT=''
REQUIRED_CONFIRMATION='RUN_20_FROZEN_CANDIDATE_CUDA_RUNS'

In [ ]:
import json, os, subprocess, sys, torch
if (REPO_DIR/'.git').is_dir(): subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone',REPO_URL,str(REPO_DIR)],check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path: sys.path.insert(0,str(REPO_DIR))
ACTIVE_COMMIT=subprocess.run(['git','rev-parse','HEAD'],check=True,capture_output=True,text=True).stdout.strip()
subprocess.run([sys.executable,'scripts/install_colab_dependencies.py','--requirements','requirements-colab.txt'],check=True)
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime > Change runtime type > GPU.')
print('Commit:',ACTIVE_COMMIT,'GPU:',torch.cuda.get_device_name(0))

## Validate candidates and the 30 reused anchor runs

Planning blocks on any source-code, dataset, preprocessing, split, feature, configuration, CUDA, completion, or test-seal incompatibility.

In [ ]:
subprocess.run([sys.executable,'scripts/run_frozen_candidate_retraining.py','--candidate-subsets',str(CANDIDATE_PATH),'--dataset-dir',str(DATASET_DIR),'--group-root',str(GROUP_ROOT),'--group-analysis-dir',str(GROUP_ANALYSIS_DIR),'--output-root',str(OUTPUT_ROOT),'--repo-dir',str(REPO_DIR)],check=True)
PLAN=json.loads((OUTPUT_ROOT/'experiment_plan.json').read_text(encoding='utf-8'))
REGISTRY=json.loads((OUTPUT_ROOT/'candidate_source_registry.json').read_text(encoding='utf-8'))
print(json.dumps(PLAN,indent=2))
print('Reused:',sum(r['source_type']=='reused_prior' for r in REGISTRY),'New:',sum(r['source_type']=='newly_trained' for r in REGISTRY))
for name,features in PLAN['frozen_candidates'].items(): print(name,len(features),features)

## Deliberate execution gate
Set `RUN_FULL_TRAINING=True` and the exact confirmation text in the configuration cell, then rerun from that cell. Only the 20 newly planned runs can execute.

In [ ]:
if not RUN_FULL_TRAINING or CONFIRMATION_TEXT != REQUIRED_CONFIRMATION:
    raise RuntimeError('Full training remains locked; review the validated 20-run plan first.')
print('Unlocked: 20 CUDA validation-only runs')

In [ ]:
from datetime import datetime, timezone
from src.colab_workflow import inspect_run_completion
from src.frozen_candidate_retraining import build_training_command, dump_json, update_registry_after_run, validate_resume_compatibility
execution=[]
for record in [r for r in REGISTRY if r['source_type']=='newly_trained']:
    run_dir=Path(record['source_run_directory']); run_dir.mkdir(parents=True,exist_ok=True)
    completion=inspect_run_completion(run_dir,record['model'])
    if completion['complete']:
        update_registry_after_run(REGISTRY,record,DATASET_DIR); action='skipped_complete'; return_code=0
    else:
        resume=validate_resume_compatibility(run_dir,record,DATASET_DIR,ACTIVE_COMMIT)
        command=build_training_command(record,DATASET_DIR,resume)
        print('Running:',' '.join(command)); result=subprocess.run(command,cwd=REPO_DIR); return_code=result.returncode; action='resumed' if resume else 'started'
        if return_code != 0: raise RuntimeError(f'Run failed: {run_dir}')
        update_registry_after_run(REGISTRY,record,DATASET_DIR)
    execution.append({'candidate':record['candidate'],'model':record['model'],'seed':record['seed'],'action':action,'return_code':return_code,'finished_at_utc':datetime.now(timezone.utc).isoformat()})
    dump_json(REGISTRY,OUTPUT_ROOT/'candidate_source_registry.json')
    dump_json(execution,OUTPUT_ROOT/'execution_manifest.json')
print('Completed or verified',len(execution),'of 20 new runs')